# Notebook 1: Building Quantum Circuits with Qforge

This notebook walks through the fundamentals of creating and manipulating quantum circuits using Qforge:

1. Creating qubits and applying gates
2. Building Bell states and GHZ states
3. Measurement and probabilities
4. Entanglement analysis
5. Parameterized rotations

In [ ]:
import numpy as np
from qforge.circuit import Qubit
from qforge.gates import H, X, Y, Z, CNOT, CCNOT, RX, RY, RZ, SWAP, Phase, S, T
from qforge.measurement import measure_all, measure_one, pauli_expectation
from qforge.data import PauliZExpectation, EntanglementEntropy

## 1. Hello Quantum World

Create a single qubit in state |0>, apply a Hadamard gate to create superposition, then measure.

In [ ]:
# Create a 1-qubit system in |0>
wf = Qubit(1)
print("Initial state |0>:")
print(f"  Amplitudes: {wf.amplitude}")
print(f"  States:     {wf.state}")

In [ ]:
# Apply Hadamard -> (|0> + |1>) / sqrt(2)
H(wf, 0)
print("After H gate:")
print(f"  Amplitudes: {wf.amplitude}")
print(f"  P(|0>) = {abs(wf.amplitude[0])**2:.4f}")
print(f"  P(|1>) = {abs(wf.amplitude[1])**2:.4f}")

In [ ]:
# Measure 1000 times
bitstrings, counts = measure_all(wf, 1000)
for bs, c in zip(bitstrings, counts):
    print(f"  |{bs}> : {c} times ({c/10:.1f}%)")

## 2. Bell State

The Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ is the simplest entangled state.
It's created by applying H on qubit 0 then CNOT(0, 1).

In [ ]:
wf = Qubit(2)
H(wf, 0)
CNOT(wf, 0, 1)

print("Bell state |Phi+>:")
for s, amp in zip(wf.state, wf.amplitude):
    if abs(amp) > 1e-10:
        print(f"  |{s}> : {amp:.4f}")

# Verify entanglement
ee = EntanglementEntropy(wf)
S_ent = ee.entanglement_entropy(bipartition=[0])
print(f"\nEntanglement entropy (qubit 0 vs qubit 1): {S_ent:.4f} bits")
print("(1.0 = maximally entangled for 2 qubits)")

In [ ]:
# Measure Bell state: always get 00 or 11, never 01 or 10
bitstrings, counts = measure_all(wf, 1000)
print("Bell state measurements:")
for bs, c in zip(bitstrings, counts):
    print(f"  |{bs}> : {c} times")

## 3. GHZ State

The GHZ state generalizes Bell to n qubits: $|GHZ\rangle = \frac{1}{\sqrt{2}}(|00...0\rangle + |11...1\rangle)$

In [ ]:
n_qubits = 4
wf = Qubit(n_qubits)

# H on first qubit, then cascade CNOT
H(wf, 0)
for i in range(n_qubits - 1):
    CNOT(wf, i, i + 1)

print(f"{n_qubits}-qubit GHZ state:")
for s, amp in zip(wf.state, wf.amplitude):
    if abs(amp) > 1e-10:
        print(f"  |{s}> : {amp:.4f}")

# Correlations: all pairs should be perfectly correlated
pz = PauliZExpectation(wf)
print(f"\nPauli-Z correlations:")
print(f"  <Z0> = {pz.one_body(0):.4f} (zero = balanced superposition)")
print(f"  <Z0 Z1> = {pz.two_body(0, 1):.4f} (one = perfectly correlated)")
print(f"  <Z0 Z3> = {pz.two_body(0, 3):.4f}")

## 4. Quantum Teleportation

Teleport an arbitrary qubit state from Alice (qubit 0) to Bob (qubit 2) using a Bell pair and classical communication.

In [ ]:
# Prepare the state to teleport: RY(pi/3)|0> on qubit 0
angle = np.pi / 3
expected_amp = np.array([np.cos(angle / 2), np.sin(angle / 2)])
print(f"State to teleport: {expected_amp[0]:.4f}|0> + {expected_amp[1]:.4f}|1>")

# Create 3-qubit system
wf = Qubit(3)

# Step 1: Prepare state on qubit 0
RY(wf, 0, angle)

# Step 2: Create Bell pair between qubits 1 and 2
H(wf, 1)
CNOT(wf, 1, 2)

# Step 3: Alice's Bell measurement (qubits 0, 1)
CNOT(wf, 0, 1)
H(wf, 0)

# Step 4: Measure qubits 0 and 1
p0 = measure_one(wf, 0)  # [P(0), P(1)]
p1 = measure_one(wf, 1)

# The state of qubit 2 now holds the teleported information
# (up to Pauli corrections based on measurement outcomes)
p2 = measure_one(wf, 2)
print(f"\nQubit 2 probabilities: P(|0>)={p2[0]:.4f}, P(|1>)={p2[1]:.4f}")
print(f"Expected:              P(|0>)={expected_amp[0]**2:.4f}, P(|1>)={expected_amp[1]**2:.4f}")

## 5. Parameterized Circuit

Sweep a rotation angle and observe how measurement probabilities change continuously.

In [ ]:
import matplotlib.pyplot as plt

angles = np.linspace(0, 2 * np.pi, 50)
probs_0 = []
expectations_z = []

for theta in angles:
    wf = Qubit(1)
    RY(wf, 0, theta)
    p = measure_one(wf, 0)
    probs_0.append(p[0])
    expectations_z.append(pauli_expectation(wf, 0, 'Z'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(angles, probs_0)
ax1.set_xlabel('RY angle (radians)')
ax1.set_ylabel('P(|0>)')
ax1.set_title('Probability of |0> vs RY angle')
ax1.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax1.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])

ax2.plot(angles, expectations_z, color='orange')
ax2.set_xlabel('RY angle (radians)')
ax2.set_ylabel('<Z>')
ax2.set_title('Pauli-Z expectation vs RY angle')
ax2.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax2.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])

plt.tight_layout()
plt.show()

## 6. Entanglement Entropy Scaling

Compare entanglement entropy of product states vs GHZ states vs random circuits.

In [ ]:
qubit_range = range(2, 8)
entropy_product = []
entropy_ghz = []
entropy_random = []

np.random.seed(42)

for n in qubit_range:
    half = list(range(n // 2))

    # Product state: no entanglement
    wf_prod = Qubit(n)
    for i in range(n):
        H(wf_prod, i)
    ee = EntanglementEntropy(wf_prod)
    entropy_product.append(ee.entanglement_entropy(bipartition=half))

    # GHZ state
    wf_ghz = Qubit(n)
    H(wf_ghz, 0)
    for i in range(n - 1):
        CNOT(wf_ghz, i, i + 1)
    ee = EntanglementEntropy(wf_ghz)
    entropy_ghz.append(ee.entanglement_entropy(bipartition=half))

    # Random circuit: high entanglement
    wf_rand = Qubit(n)
    for _ in range(3):  # 3 layers
        for i in range(n):
            RY(wf_rand, i, np.random.uniform(0, 2 * np.pi))
            RZ(wf_rand, i, np.random.uniform(0, 2 * np.pi))
        for i in range(n - 1):
            CNOT(wf_rand, i, i + 1)
    ee = EntanglementEntropy(wf_rand)
    entropy_random.append(ee.entanglement_entropy(bipartition=half))

plt.figure(figsize=(8, 5))
plt.plot(list(qubit_range), entropy_product, 'o-', label='Product state (H on each)')
plt.plot(list(qubit_range), entropy_ghz, 's-', label='GHZ state')
plt.plot(list(qubit_range), entropy_random, '^-', label='Random circuit (3 layers)')
plt.xlabel('Number of qubits')
plt.ylabel('Entanglement entropy (bits)')
plt.title('Entanglement Entropy Scaling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()